In [216]:
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import make_moons
from tqdm.auto import tqdm
from torch import Tensor, nn
import torchode as to

In [217]:
class Flow(nn.Module):
    def __init__(self, dim: int = 2, h: int = 64):
        super().__init__()
        self.net = nn.GRU(dim + 1, h, num_layers=2, batch_first=True)
        self.proj = nn.Linear(h, 1)

    def forward(self, t: Tensor, x_t: Tensor) -> Tensor:
        t = t.unsqueeze(-1).expand(x_t.shape[0], x_t.shape[1], 1)
        h_t = self.net(torch.cat([x_t, t], dim=-1))[0]  # (B, L, C)
        u_t = self.proj(h_t).squeeze(-1)
        return u_t

    def step(self, x_t: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor:
        t_start = t_start.view(1, 1).expand(x_t.shape[0], 1)

        return x_t + (t_end - t_start) * self(
            t=t_start + (t_end - t_start) / 2,
            x_t=x_t + self(x_t=x_t, t=t_start) * (t_end - t_start) / 2,
        )


In [218]:
# Training
flow = Flow(dim=1, h=64)

optimizer = torch.optim.Adam(flow.parameters(), 1e-2)
loss_fn = nn.MSELoss()

tau = torch.linspace(0, 1, 100)
for i in tqdm(range(1000)):
    freq = torch.rand(256) * torch.pi
    x_1 = torch.sin(tau[None, :] * freq[:, None] + torch.pi)
    x_0 = torch.randn_like(x_1).cumsum(dim=1)
    t = torch.rand(len(x_1), 1)

    x_t = (1 - t) * x_0 + t * x_1
    dx_t = x_1 - x_0

    optimizer.zero_grad()
    loss = loss_fn(flow(t=t, x_t=x_t.unsqueeze(-1)), dx_t)
    loss.backward()
    optimizer.step()
    if i % 100 == 0:
        print(loss)

  0%|          | 0/1000 [00:00<?, ?it/s]

tensor(52.4042, grad_fn=<MseLossBackward0>)
tensor(3.5109, grad_fn=<MseLossBackward0>)
tensor(1.8993, grad_fn=<MseLossBackward0>)
tensor(5.1548, grad_fn=<MseLossBackward0>)


KeyboardInterrupt: 

In [ ]:
# Sampling
x = torch.randn(300, 2)
n_steps = 8
fig, axes = plt.subplots(1, n_steps + 1, figsize=(30, 4), sharex=True, sharey=True)
time_steps = torch.linspace(0, 1.0, n_steps + 1)

axes[0].scatter(x.detach()[:, 0], x.detach()[:, 1], s=10)
axes[0].set_title(f"t = {time_steps[0]:.2f}")
axes[0].set_xlim(-3.0, 3.0)
axes[0].set_ylim(-3.0, 3.0)

for i in range(n_steps):
    x = flow.step(x_t=x, t_start=time_steps[i], t_end=time_steps[i + 1])
    axes[i + 1].scatter(x.detach()[:, 0], x.detach()[:, 1], s=10)
    axes[i + 1].set_title(f"t = {time_steps[i + 1]:.2f}")

plt.tight_layout()
plt.show()
